# Paintings → Structured Semantics → Speech (LOCAL)

> **Local version** — runs on your own machine. The Colab version is `02_semantics_to_speech.ipynb`.

**Assumes notebook 01 (`01_image_to_text_finetuning_local`) has already produced a fine-tuned BLIP checkpoint.**

This notebook:
1. Loads the fine-tuned BLIP captioner.
2. Generates **structured semantics** (title, caption, year, period, `suggested_tone`) for each painting.
3. Synthesises a **spoken description** with `edge-tts`, picking the voice and prosody style
   automatically from `suggested_tone` (`"calm"` or `"engaging"`).

Output: one `.mp3` per painting saved to `DATA_DIR/tts_outputs/`.

## Prerequisites

Assumes the `lavis` conda environment with LAVIS and all dependencies already installed.
Also requires `edge-tts` and `nest_asyncio`:

```
pip install edge-tts nest_asyncio
```

Run this notebook **from the notebooks folder** (`mscai-multimodal-project/notebooks`):

```
conda activate lavis
jupyter notebook
```

In [ ]:
# Local paths — uses the same OUTPUTS_DIR as notebook 01.
DATA_DIR    = "../data"
OUTPUTS_DIR = "../outputs"   # shared outputs root (mirrors notebook 01)

# Auto-detect the latest fine-tuned checkpoint produced by notebook 01.
import glob, os
run_dirs = sorted(glob.glob(OUTPUTS_DIR + "/train_*"))
if run_dirs:
    CHECKPOINT_PATH = os.path.join(run_dirs[-1], "blip_artpedia_best.pth")
else:
    CHECKPOINT_PATH = None  # no training run found — run notebook 01 first

print(f"DATA_DIR        = {DATA_DIR}")
print(f"CHECKPOINT_PATH = {CHECKPOINT_PATH}")

## 1. Load Fine-tuned Captioner

Loads the fine-tuned BLIP captioner from the local checkpoint.

In [2]:
import sys
sys.path.insert(0, '../src')

from captioner import Captioner

captioner = Captioner(checkpoint_path=CHECKPOINT_PATH)
print("Captioner ready.")

c:\Users\Elpida\anaconda3\envs\lavis_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Elpida\anaconda3\envs\lavis_clean\lib\site-packages\fairscale\experimental\nn\offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
c:\Users\Elpida\anaconda3\envs\lavis_clean\lib\site-packages\fairscale\experimental\nn\offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore


Loaded checkpoint: ../data/checkpoints/blip_artpedia_epoch5.pth
  missing keys: 0  unexpected keys: 0
Captioner ready.


## 2. Generate Structured Semantics

For each of the first `N` paintings in the test manifest:
1. Loads the cached image from disk (local path from manifest — no network).
2. Generates a caption with the fine-tuned captioner.
3. Calls `build_semantics(entry, caption)` — the manifest entry has `title` and `year`, which is all that's needed.

> **Note:** `year` is included in the manifest from `prepare_data.py`. If your existing manifest
> was built before this field was added, re-run:
> ```
> python src/prepare_data.py --split test --output-dir "../data" --rebuild-manifest
> ```

In [3]:
import json
from pathlib import Path
from PIL import Image as PILImage
from structured_semantics import build_semantics

N = 5  # number of paintings to process; increase as needed

all_semantics = []

# The manifest has image_path, caption, title, and year — everything build_semantics needs.
manifest_path = Path(DATA_DIR) / "processed" / "test.jsonl"
with open(manifest_path, encoding="utf-8-sig") as f:
    manifest_entries = [json.loads(line) for line in f if line.strip()]

for entry in manifest_entries[:N]:
    # Load image from local cached path — no network call.
    img_path = Path(DATA_DIR) / entry["image_path"]
    image = PILImage.open(img_path).convert("RGB")

    caption = captioner.caption_image(image, num_captions=1)[0]

    # Pass the manifest entry directly as the record — it has title and year.
    sem = build_semantics(entry, caption)
    all_semantics.append(sem)

    print(json.dumps(sem, indent=2, ensure_ascii=False))
    print()

{
  "title": "Profile Portrait of a Lady",
  "caption": "portrait of a young woman is a c 1305 oil on panel painting by the italian renaissance artist sandro bottic, now in",
  "year": 1410,
  "period": "Renaissance",
  "suggested_tone": "calm",
  "source": ""
}

{
  "title": "San Pietro Martire Triptych",
  "caption": "the central panel shows the virgin and child surrounded by saints and angels the central panel shows the virgin and child surrounded by saints and angels",
  "year": 1420,
  "period": "Renaissance",
  "suggested_tone": "calm",
  "source": ""
}

{
  "title": "Annunciation (Lochner)",
  "caption": "the painting depicts the virgin and the angel, with a book in their lap, on the left, and on the right, the",
  "year": 1430,
  "period": "Renaissance",
  "suggested_tone": "calm",
  "source": ""
}

{
  "title": "Perugia Altarpiece",
  "caption": "the central panel shows the virgin and child surrounded by the saints, the virgin mary and the child john the baptist, the virgin ma

## 3. TTS Setup

Two prosody styles matching the project's TTS spec:
- **calm** — `en-GB-SoniaNeural`, slower and lower-pitched (classical/older works).
- **engaging** — `en-US-JennyNeural`, warmer and slightly elevated pitch (dynamic works).

Output mp3s are saved to `DATA_DIR/tts_outputs/`.

In [5]:
# Windows SSL cert-store workaround.
# aiohttp (used by edge_tts) calls ssl.create_default_context() at import time,
# which bundles ALL Windows-store certs into one blob. One malformed cert makes
# load_verify_locations() fail with [ASN1: NOT_ENOUGH_DATA].
# Fix: patch _load_windows_store_certs to load each cert individually and skip bad ones.
import ssl

_orig_load_windows_store_certs = ssl.SSLContext._load_windows_store_certs

def _patched_load_windows_store_certs(self, storename, purpose):
    loaded = skipped = 0
    try:
        for cert, encoding, trust in ssl.enum_certificates(storename):
            if encoding == "x509_asn" and (trust is True or purpose.oid in trust):
                try:
                    self.load_verify_locations(cadata=cert)
                    loaded += 1
                except ssl.SSLError:
                    skipped += 1
    except PermissionError:
        pass
    if skipped:
        print(f"[SSL patch] store='{storename}': loaded {loaded}, skipped {skipped} malformed cert(s)")

ssl.SSLContext._load_windows_store_certs = _patched_load_windows_store_certs
print("SSL patch applied.")

SSL patch applied.


In [6]:
import edge_tts
import nest_asyncio
import asyncio
import os

nest_asyncio.apply()  # lets `await` work inside Jupyter's existing event loop

TTS_OUTPUT_DIR = DATA_DIR + "/tts_outputs"
os.makedirs(TTS_OUTPUT_DIR, exist_ok=True)
print(f"TTS output dir: {TTS_OUTPUT_DIR}")

# Tone definitions — strings must match suggested_tone values: "calm" / "engaging".
STYLES = {
    "calm": {
        "voice":  "en-GB-SoniaNeural",
        "rate":   "-5%",
        "pitch":  "-2Hz",
        "volume": "+0%",
    },
    "engaging": {
        "voice":  "en-US-JennyNeural",
        "rate":   "-2%",
        "pitch":  "+4Hz",
        "volume": "+3%",
    },
}

[SSL patch] store='CA': loaded 0, skipped 11 malformed cert(s)
[SSL patch] store='ROOT': loaded 0, skipped 46 malformed cert(s)
TTS output dir: ../data/tts_outputs


## 4. Synthesise Speech

`build_spoken_text` composes the spoken text from the semantics dict:
it prepends an optional intro (`<Title>. <Period>, circa <year>.`) built only from
fields that are present and meaningful, then appends the generated caption.
If no metadata intro can be built, only the caption is spoken.

Tone selection from `suggested_tone` and all voice/prosody settings are unchanged.

In [7]:
def build_spoken_text(sem: dict) -> str:
    """Compose spoken text: optional metadata intro + caption."""
    title  = sem.get("title", "")
    period = sem.get("period", "")
    year   = sem.get("year")

    intro_parts = []
    if title and title != "Untitled":
        intro_parts.append(title + ".")

    meta = []
    if period and period != "Unknown":
        meta.append(period)
    if year is not None:
        meta.append(f"circa {year}")
    if meta:
        intro_parts.append(", ".join(meta) + ".")

    intro   = " ".join(intro_parts)
    caption = sem.get("caption", "")
    return f"{intro} {caption}".strip() if intro else caption


async def synthesise_all(semantics_list):
    for idx, sem in enumerate(semantics_list):
        # Tone is set by the semantics pipeline; fall back to calm if unrecognised.
        tone = sem.get("suggested_tone", "calm")
        if tone not in STYLES:
            print(f"  [WARN] Unknown tone '{tone}' for index {idx} — using 'calm'.")
            tone = "calm"

        style    = STYLES[tone]
        out_path = os.path.join(TTS_OUTPUT_DIR, f"{idx}_{tone}.mp3")

        communicate = edge_tts.Communicate(
            text=build_spoken_text(sem),
            voice=style["voice"],
            rate=style["rate"],
            pitch=style["pitch"],
            volume=style["volume"],
        )
        await communicate.save(out_path)
        print(f"  [{tone}] saved: {out_path}")


await synthesise_all(all_semantics)

  [calm] saved: ../data/tts_outputs\0_calm.mp3
  [calm] saved: ../data/tts_outputs\1_calm.mp3
  [calm] saved: ../data/tts_outputs\2_calm.mp3
  [calm] saved: ../data/tts_outputs\3_calm.mp3
  [calm] saved: ../data/tts_outputs\4_calm.mp3


## 5. Play a Sample Inline (Optional)

Change `PLAY_INDEX` to listen to any of the generated files directly in the notebook.

In [8]:
from IPython.display import Audio

PLAY_INDEX = 0
play_tone  = all_semantics[PLAY_INDEX].get("suggested_tone", "calm")
mp3_path   = os.path.join(TTS_OUTPUT_DIR, f"{PLAY_INDEX}_{play_tone}.mp3")

print(f"Playing: {mp3_path}")
Audio(mp3_path)

Playing: ../data/tts_outputs\0_calm.mp3
